# Solución 1: Clasificación de Riesgo de Cancelación con Random Forest
* **Objetivo de Negocio**: Predecir el riesgo de cancelación de un pedido en el momento en que se registra en el e-commerce de Distribuidora Panamericana para tomar acciones preventivas antes del despacho.
* **Unidad de Análisis**: Un pedido de compra individual completo.
* **Variable Objetivo (`clase_y`)**: Binaria (`1 = Cancelado`, `0 = Entregado`).
* **Origen de Datos**: Repositorio GitHub en línea (`pryBinaBack/outputs/dataset_clasificacion.csv`).


## 1. Dependencias e importación de librerías


In [ ]:
import os
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib

from sklearn.feature_extraction import DictVectorizer
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import StratifiedKFold, cross_validate, train_test_split
from sklearn.metrics import (
    make_scorer, classification_report, confusion_matrix, accuracy_score,
    average_precision_score, balanced_accuracy_score,
    precision_score, recall_score, f1_score
)
from sklearn.pipeline import Pipeline

try:
    from IPython.display import display
except ImportError:
    display = print


## 2. Carga del Dataset (En línea desde GitHub con Respaldo Local)


In [ ]:
url_github = "https://raw.githubusercontent.com/Oscar-David-Dela-Cruz-Hdez/pryBinaBack/master/outputs/dataset_clasificacion.csv"

df = pd.read_csv(url_github)

print("Dimensiones del dataset cargado (Filas, Columnas):", df.shape)
display(df.head())

## 3. Análisis Exploratorio de Datos (EDA) y Control de Calidad


In [ ]:
print("=== REVISIÓN DE VALORES NULOS Y TIPOS DE DATOS ===")
print("Porcentaje de valores nulos por columna (%):")
display(df.isnull().mean() * 100)

print("\n=== DISTRIBUCIÓN DE LA VARIABLE OBJETIVO (clase_y) ===")
conteo_clases = df["clase_y"].value_counts().rename({0: "Entregado (0)", 1: "Cancelado (1)"})
display(conteo_clases.to_frame("Cantidad"))

plt.figure(figsize=(6, 4))
sns.countplot(data=df, x="clase_y", palette="Set2")
plt.xticks([0, 1], ["Entregado (0)", "Cancelado (1)"])
plt.title("Distribución de Pedidos Entregados vs Cancelados")
plt.xlabel("Estado del Pedido (clase_y)")
plt.ylabel("Número de Pedidos")
plt.show()


## 4. Preparación de Datos e Ingeniería de Características (Feature Engineering)


In [ ]:
X_dict = []
for _, row in df.iterrows():
    total = float(row["total"])
    costo_envio = float(row["costo_envio"])
    num_productos = float(row["num_productos"]) if float(row["num_productos"]) > 0 else 1.0
    total_unidades = float(row["total_unidades"]) if float(row["total_unidades"]) > 0 else 1.0

    fila = {
        "edad": float(row["edad"]) if pd.notnull(row["edad"]) else 0.0,
        "total": total,
        "costo_envio": costo_envio,
        "porcentaje_cancelados_previos": float(row["porcentaje_cancelados_previos"]),
        "metodo_pago": str(row["metodo_pago"]) if pd.notnull(row["metodo_pago"]) else "Sin definir",
        "usuario_id": str(row["usuario_id"]),
        "num_productos": num_productos,
        "total_unidades": total_unidades,
        "hora_compra": float(row["hora_compra"]) if "hora_compra" in row and pd.notnull(row["hora_compra"]) else 12.0,
        "es_fin_de_semana": float(row["es_fin_de_semana"]) if "es_fin_de_semana" in row and pd.notnull(row["es_fin_de_semana"]) else 0.0,
        "dias_desde_ultimo_pedido": float(row["dias_desde_ultimo_pedido"]) if "dias_desde_ultimo_pedido" in row and pd.notnull(row["dias_desde_ultimo_pedido"]) else 999.0,
        "antiguedad_cuenta_dias": float(row["antiguedad_cuenta_dias"]) if "antiguedad_cuenta_dias" in row and pd.notnull(row["antiguedad_cuenta_dias"]) else 0.0,
        "estado_envio": str(row["estado_envio"]) if "estado_envio" in row and pd.notnull(row["estado_envio"]) else "Hidalgo",
        "precio_promedio_unidad": total / total_unidades,
        "unidades_por_producto": total_unidades / num_productos,
        "prop_costo_envio": costo_envio / (total if total > 0 else 1.0),
        "envio_gratis": 1.0 if costo_envio == 0 else 0.0,
        "total_real": total + costo_envio
    }
    X_dict.append(fila)

y = df["clase_y"].astype(int).values
print("Total de observaciones vectorizadas:", len(X_dict))


## 5. Modelado y Evaluación con Validación Cruzada (Stratified K-Fold = 5)


In [ ]:
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

pipeline = Pipeline([
    ("vectorizacion", DictVectorizer(sparse=True)),
    ("clasificador", RandomForestClassifier(
        n_estimators=300,
        max_depth=12,
        min_samples_split=10,
        min_samples_leaf=4,
        class_weight="balanced",
        random_state=42,
        n_jobs=-1
    ))
])

scorers = {
    "accuracy": "accuracy",
    "balanced_accuracy": "balanced_accuracy",
    "f1": "f1",
    "precision": "precision",
    "recall": "recall",
    "pr_auc": make_scorer(average_precision_score, response_method="predict_proba")
}

cv_results = cross_validate(pipeline, X_dict, y, cv=skf, scoring=scorers)

df_kfold = pd.DataFrame({
    "Fold": [f"Fold {i+1}" for i in range(5)],
    "Accuracy": cv_results["test_accuracy"].round(3),
    "Balanced Accuracy": cv_results["test_balanced_accuracy"].round(3),
    "Precisión": cv_results["test_precision"].round(3),
    "Recall": cv_results["test_recall"].round(3),
    "F1-Score": cv_results["test_f1"].round(3),
    "PR-AUC": cv_results["test_pr_auc"].round(3)
})

print("=== MÉTRICAS POR PLIEGUE (FOLD 1 A 5) ===")
display(df_kfold)

df_promedios = pd.DataFrame({
    "Métrica": ["Accuracy", "Balanced Accuracy", "Precisión", "Recall", "F1-Score", "PR-AUC"],
    "Promedio (Media)": [
        cv_results["test_accuracy"].mean().round(3),
        cv_results["test_balanced_accuracy"].mean().round(3),
        cv_results["test_precision"].mean().round(3),
        cv_results["test_recall"].mean().round(3),
        cv_results["test_f1"].mean().round(3),
        cv_results["test_pr_auc"].mean().round(3)
    ],
    "Desviación Estándar (±)": [
        cv_results["test_accuracy"].std().round(3),
        cv_results["test_balanced_accuracy"].std().round(3),
        cv_results["test_precision"].std().round(3),
        cv_results["test_recall"].std().round(3),
        cv_results["test_f1"].std().round(3),
        cv_results["test_pr_auc"].std().round(3)
    ],
    "Media ± Desv. Estándar": [
        f"{cv_results['test_accuracy'].mean():.3f} ± {cv_results['test_accuracy'].std():.3f}",
        f"{cv_results['test_balanced_accuracy'].mean():.3f} ± {cv_results['test_balanced_accuracy'].std():.3f}",
        f"{cv_results['test_precision'].mean():.3f} ± {cv_results['test_precision'].std():.3f}",
        f"{cv_results['test_recall'].mean():.3f} ± {cv_results['test_recall'].std():.3f}",
        f"{cv_results['test_f1'].mean():.3f} ± {cv_results['test_f1'].std():.3f}",
        f"{cv_results['test_pr_auc'].mean():.3f} ± {cv_results['test_pr_auc'].std():.3f}"
    ]
})

print("\n=== RESUMEN DE PROMEDIOS K-FOLD = 5 ===")
display(df_promedios)


## 6. Matriz de Confusión e Importancia de Características


In [ ]:
X_train_dict, X_test_dict, y_train, y_test = train_test_split(
    X_dict, y, test_size=0.20, random_state=42, stratify=y
)

pipeline.fit(X_train_dict, y_train)
probabilidad = pipeline.predict_proba(X_test_dict)[:, 1]
prediccion = (probabilidad >= 0.50).astype(int)

cm = confusion_matrix(y_test, prediccion)
plt.figure(figsize=(6, 4))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=["Entregado", "Cancelado"], yticklabels=["Entregado", "Cancelado"])
plt.xlabel("Predicción")
plt.ylabel("Real")
plt.title("Matriz de Confusión - Random Forest (Test Set)")
plt.show()

vectorizador = pipeline.named_steps["vectorizacion"]
bosque = pipeline.named_steps["clasificador"]
importancias = pd.Series(bosque.feature_importances_, index=vectorizador.get_feature_names_out())
print("=== VARIABLES CON MAYOR IMPORTANCIA EN EL MODELO ===")
display(importancias.sort_values(ascending=False).head(10).to_frame("Importancia"))


## 7. Entrenar Modelo Final y Guardar Artefacto (.joblib)


In [ ]:
pipeline.fit(X_dict, y)
joblib.dump(pipeline, "modelo_random_forest_cancelacion.joblib")
print("[OK] Modelo final entrenado y guardado exitosamente como modelo_random_forest_cancelacion.joblib")
